# Oracle Freshness Cache Mode

This notebook demonstrates `retrieval_mode="freshness_cache"`.

What you should see:

1. Run 1 has no fresh Oracle cache row, so it calls Tavily and persists results.
2. Run 2 uses the fresh Oracle cache and returns local results.
3. The final cell deletes this notebook's demo rows so the first run stays meaningful next time.

## Setup

Run this cell first. It loads `.env`, connects to Oracle, prepares the demo table, and imports shared notebook helpers.

In [ ]:
from pathlib import Path
import sys

helper_path = None
for root in [Path.cwd(), *Path.cwd().parents]:
    candidate = root / "examples" / "oracle" / "demo_helpers.py"
    if candidate.exists():
        helper_path = candidate
        break

if helper_path is None:
    candidate = Path.cwd() / "demo_helpers.py"
    if candidate.exists():
        helper_path = candidate

if helper_path is None:
    raise RuntimeError("Could not find examples/oracle/demo_helpers.py. Start Jupyter from the repository root or examples/oracle.")

sys.path.insert(0, str(helper_path.parent))
from demo_helpers import *

print("Using helper:", helper_path)

## Choose the demo query

Edit only `CACHE_QUERY` below when you want to try your own cache demo query.

In [ ]:
CACHE_QUERY = "Oracle Database vector search freshness cache for Tavily results"

## Start from a clean query-specific state

In [ ]:
delete_rows_for_query(CACHE_QUERY)

## Run the same query twice

The low threshold is intentional for the demo. It makes the second run visibly hit Oracle cache with the deterministic demo embeddings.

In [ ]:
cache_client = make_client(
    "freshness_cache",
    persistence_depth="cache_only",
    cache_ttl_seconds=3600,
    cache_score_threshold=-1.0,
)

first_results = cache_client.search(
    CACHE_QUERY,
    max_results=3,
    max_local=3,
    max_foreign=3,
    save_foreign=True,
    **TAVILY_SEARCH_OPTIONS,
)
show_results("Run 1: cache miss, Tavily fallback", first_results)
show_persisted_rows(CACHE_QUERY, "Oracle rows after Run 1")

second_results = cache_client.search(
    CACHE_QUERY,
    max_results=3,
    max_local=3,
    max_foreign=3,
    save_foreign=True,
    **TAVILY_SEARCH_OPTIONS,
)
show_results("Run 2: fresh Oracle cache hit", second_results)
show_persisted_rows(CACHE_QUERY, "Oracle rows after Run 2")

## Cleanup

Run this at the end. It deletes this notebook's demo rows so the next full run starts with a cache miss again.

In [ ]:
delete_rows_for_query(CACHE_QUERY)
print("Cleaned freshness-cache demo rows. Re-run from the top to see Run 1 call Tavily again.")